In [1]:
import pandas as pd
import numpy as np
# import matplotlib.pyplot as plt
# from scipy import stats
from IPython.display import display
import datetime

def export(df: pd.DataFrame, path_no_dot: str, index=True):
    df.to_csv(f'exports/{path_no_dot}.csv', index=index)
    df.to_excel(f'exports/{path_no_dot}.xlsx', index=index)

    
# THINGS TO CHANGE WHEN ADDING NEW CONTENT

# coloring and sectioning the ELO graph
vrect_colors = ['green', 'red', 'yellow', 'blue', 'orange']
# seasons = ['2021_2s', '2022_1s', '2022_2s', '2023_1s','2023_RS','2023_RS_BW']
# season_labels = ['2021<br>Doubles', '2022<br>Singles', '2022<br>Doubles', '2023<br>Singles','2023<br>Regular<br>Season', '2023<br>Beach<br>Week']
seasons = ['2021_2s', '2022_1s', '2022_2s', '2023_1s','2023_2s','2024_1s','2024_2s','2025_1s','2025_2s']
season_labels = ['2021<br>Doubles', '2022<br>Singles', '2022<br>Doubles', '2023<br>Singles','2023<br>Doubles','2024<br>Singles','2024<br>Doubles','2025<br>Singles','2025<br>Doubles']

In [2]:
# data initialization

# needs ppt_analysis.ipynb to be ran first
players_table = pd.read_csv('data/players.csv')
matches_table = pd.read_csv('data/matches.csv').sort_values(['event','match_number']).reset_index(drop=True)
matches_table = matches_table.loc[:,matches_table.columns != 'notes']
# matches_table['date'] = pd.to_datetime(matches_table['date'])
matches_table = matches_table.loc[matches_table.event != '2023_RS',:]
matches_table = matches_table.loc[matches_table.event != '2023_RS_BW',:]

# with pd.option_context('display.max_rows', None, 'display.max_columns', None):
#     display(matches_table)

In [3]:
def expected_score(ratingA, ratingB): # expected difference in cups hit
    # sigmoid vertical bounds
    yrange = 12
    ymin = 6
    # game variables
    expected_cups = 2
    normal_elo_difference = 400

    # score is the expected difference in cups scored,
        # if this function spits out -5, then players of those two ratings should end in a difference of 5 cups
    # this function is a sigmoid function from -6 to 6
    # this function says an elo difference of "normal_elo_difference" should result in absolute difference in "expected_cups"
        # this works: an elo difference of 400 should result in a score difference of 2
    return (yrange / (1 + np.power(expected_cups, (ratingB - ratingA) / normal_elo_difference))) - ymin

def rating_change(score, expected_score):
    # the k-factor: determines how strongly a result affects the rating change
    # usually between 10 and 40, but with few games, we want to change it frequently
    K = 12
    return K * (score - expected_score)

def expected_score_inverse(expected_cups):
    yrange = 12
    ymin = -6
    normal_elo_change = 400
    # log base 2 shows difference in 400 rating make a score difference of 2
    expected_difference_in_elo = normal_elo_change * np.log2((yrange / (expected_cups - ymin)) - 1)
    return expected_difference_in_elo

def expected_win_prob(ratingA, ratingB):
    # game variables
    base = 2
    normal_elo_difference = 100

    return (1 / (1 + np.power(base, (ratingB - ratingA) / normal_elo_difference)))


# ELO INITIALIZATION
starting_elo = 1200.0
elo = dict()
prev_elo_time = np.empty([len(players_table['player']), len(matches_table) + 1])
prev_other_elo_time = np.zeros([len(players_table['player']), len(matches_table) + 1])
elo_time = np.zeros([len(players_table['player']), len(matches_table) + 1])
d_elo_time = np.empty([len(players_table['player']), len(matches_table) + 1])
ec_time = np.empty([len(players_table['player']), len(matches_table) + 1])
caa_time = np.empty([len(players_table['player']), len(matches_table) + 1])
wp_time = np.empty([len(players_table['player']), len(matches_table) + 1])


for player in players_table['player']:
    elo.update({player: starting_elo})
    prev_elo_time[players_table['player'][players_table['player'] == player].index[0], 0] = starting_elo
    prev_other_elo_time[players_table['player'][players_table['player'] == player].index[0], 0] = starting_elo
    elo_time[players_table['player'][players_table['player'] == player].index[0], 0] = starting_elo
    caa_time[players_table['player'][players_table['player'] == player].index[0], 0] = starting_elo
    wp_time[players_table['player'][players_table['player'] == player].index[0], 0] = 0


# ALGORITHM
prev_t1_row = 1
prev_t2_row = 1

for (i, row) in matches_table.iterrows():
    elos_t1 = list()
    elos_t2 = list()

    elos_t1.append(elo[row.t1p1])
    elos_t2.append(elo[row.t2p1])

    # doubles
    if (type(row.t1p2) == str) and (type(row.t2p2) == str):
        elos_t1.append(elo[row.t1p2])
        elos_t2.append(elo[row.t2p2])
        
        prev_elo_time[players_table['player'][players_table['player'] == row.t1p2].index[0], i + 1] = elo[row.t1p2]
        prev_elo_time[players_table['player'][players_table['player'] == row.t2p2].index[0], i + 1] = elo[row.t2p2]

    elo_t1 = np.mean(elos_t1)
    elo_t2 = np.mean(elos_t2)

    # NEW CODE
    prev_elo_time[players_table['player'][players_table['player'] == row.t1p1].index[0], i + 1] = elo_t1
    prev_elo_time[players_table['player'][players_table['player'] == row.t2p1].index[0], i + 1] = elo_t2
    
    prev_other_elo_time[players_table['player'][players_table['player'] == row.t1p1].index[0], i + 1] = elo_t2
    prev_other_elo_time[players_table['player'][players_table['player'] == row.t2p1].index[0], i + 1] = elo_t1

    
    # doubles
    if (type(row.t1p2) == str) and (type(row.t2p2) == str):
        prev_other_elo_time[players_table['player'][players_table['player'] == row.t1p2].index[0], i + 1] = elo_t2
        prev_other_elo_time[players_table['player'][players_table['player'] == row.t2p2].index[0], i + 1] = elo_t1

    win_prob_t1 = expected_score(elo_t1, elo_t2)
    win_prob_t2 = expected_score(elo_t2, elo_t1)
    
    wp_time[players_table['player'][players_table['player'] == row.t1p1].index[0], i + 1] = expected_win_prob(elo_t1, elo_t2)
    wp_time[players_table['player'][players_table['player'] == row.t2p1].index[0], i + 1] = expected_win_prob(elo_t2, elo_t1)

    rating_change_t1p1 = rating_change(row.score1 - row.score2, win_prob_t1)
    rating_change_t2p1 = rating_change(row.score2 - row.score1, win_prob_t2)

    elo[row.t1p1] += rating_change_t1p1
    elo[row.t2p1] += rating_change_t2p1    
    
    d_elo_time[players_table['player'][players_table['player'] == row.t1p1].index[0], i + 1] = rating_change_t1p1
    d_elo_time[players_table['player'][players_table['player'] == row.t2p1].index[0], i + 1] = rating_change_t2p1
    
    ec_time[players_table['player'][players_table['player'] == row.t1p1].index[0], i + 1] = expected_score(elo_t1, elo_t2)
    ec_time[players_table['player'][players_table['player'] == row.t2p1].index[0], i + 1] = expected_score(elo_t2, elo_t1)

    caa_time[players_table['player'][players_table['player'] == row.t1p1].index[0], i + 1] = expected_score(elo_t1, starting_elo)
    caa_time[players_table['player'][players_table['player'] == row.t2p1].index[0], i + 1] = expected_score(elo_t2, starting_elo)

    elo_time[players_table['player'][players_table['player'] == row.t1p1].index[0], i + 1] = elo[row.t1p1]
    elo_time[players_table['player'][players_table['player'] == row.t2p1].index[0], i + 1] = elo[row.t2p1]

    # doubles
    if (type(row.t1p2) == str) and (type(row.t2p2) == str):

        wp_time[players_table['player'][players_table['player'] == row.t1p2].index[0], i + 1] = expected_win_prob(elo_t1, elo_t2)
        wp_time[players_table['player'][players_table['player'] == row.t2p2].index[0], i + 1] = expected_win_prob(elo_t2, elo_t1)

        rating_change_t1p2 = rating_change(row.score1 - row.score2, win_prob_t1)
        rating_change_t2p2 = rating_change(row.score2 - row.score1, win_prob_t2)

        elo[row.t1p2] += rating_change_t1p2
        elo[row.t2p2] += rating_change_t2p2

        d_elo_time[players_table['player'][players_table['player'] == row.t1p2].index[0], i + 1] = rating_change_t1p2
        d_elo_time[players_table['player'][players_table['player'] == row.t2p2].index[0], i + 1] = rating_change_t2p2

        ec_time[players_table['player'][players_table['player'] == row.t1p2].index[0], i + 1] = expected_score(elo_t1, elo_t2)
        ec_time[players_table['player'][players_table['player'] == row.t2p2].index[0], i + 1] = expected_score(elo_t2, elo_t1)

        caa_time[players_table['player'][players_table['player'] == row.t1p2].index[0], i + 1] = expected_score(elo_t1, starting_elo)
        caa_time[players_table['player'][players_table['player'] == row.t2p2].index[0], i + 1] = expected_score(elo_t2, starting_elo)
        
        elo_time[players_table['player'][players_table['player'] == row.t1p2].index[0], i + 1] = elo[row.t1p2]
        elo_time[players_table['player'][players_table['player'] == row.t2p2].index[0], i + 1] = elo[row.t2p2]

players_table['current_elo'] = players_table['player'].map(elo)
players_table['current_elo'] = players_table['current_elo'].apply('{:.0f}'.format)

player_elo_table = players_table[['player', 'current_elo']].sort_values('current_elo', ascending=False).reset_index(drop=True)

# with pd.option_context('display.max_rows', 100, 'display.max_columns', None):
#     display(players_table)

export(player_elo_table, 'player_elo_table', False)

In [4]:
print("Expected score between 1400 and 1000", expected_score(1400, 1000))
print()
print("A much higher ELO, B barely wins: A's change is", rating_change(5 - 6, expected_score(1400, 1000)))
print("A much higher ELO, A barely wins: A's change is", rating_change(6 - 5, expected_score(1400, 1000)))
print("A much higher ELO, B hugely wins: A's change is", rating_change(1 - 6, expected_score(1400, 1000)))
print("A much higher ELO, A hugely wins: A's change is", rating_change(6 - 1, expected_score(1400, 1000)))
print()
print("A closely higher ELO, B barely wins: A's change is", rating_change(5 - 6, expected_score(1250, 1200)))
print("A closely higher ELO, A barely wins: A's change is", rating_change(6 - 5, expected_score(1250, 1200)))
print("A closely higher ELO, B hugely wins: A's change is", rating_change(1 - 6, expected_score(1250, 1200)))
print("A closely higher ELO, A hugely wins: A's change is", rating_change(6 - 1, expected_score(1250, 1200)))
print()
print("A closely lower ELO, B barely wins: A's change is", rating_change(5 - 6, expected_score(1200, 1250)))
print("A closely lower ELO, A barely wins: A's change is", rating_change(6 - 5, expected_score(1200, 1250)))
print("A closely lower ELO, B hugely wins: A's change is", rating_change(1 - 6, expected_score(1200, 1250)))
print("A closely lower ELO, A hugely wins: A's change is", rating_change(6 - 1, expected_score(1200, 1250)))
print()
print("A much lower ELO, B barely wins: A's change is", rating_change(5 - 6, expected_score(1000, 1400)))
print("A much lower ELO, A barely wins: A's change is", rating_change(6 - 5, expected_score(1000, 1400)))
print("A much lower ELO, B hugely wins: A's change is", rating_change(1 - 6, expected_score(1000, 1400)))
print("A much lower ELO, A hugely wins: A's change is", rating_change(6 - 1, expected_score(1000, 1400)))
print()

Expected score between 1400 and 1000 2.0

A much higher ELO, B barely wins: A's change is -36.0
A much higher ELO, A barely wins: A's change is -12.0
A much higher ELO, B hugely wins: A's change is -84.0
A much higher ELO, A hugely wins: A's change is 36.0

A closely higher ELO, B barely wins: A's change is -15.117212459956022
A closely higher ELO, A barely wins: A's change is 8.882787540043978
A closely higher ELO, B hugely wins: A's change is -63.11721245995602
A closely higher ELO, A hugely wins: A's change is 56.88278754004398

A closely lower ELO, B barely wins: A's change is -8.882787540043967
A closely lower ELO, A barely wins: A's change is 15.117212459956033
A closely lower ELO, B hugely wins: A's change is -56.882787540043964
A closely lower ELO, A hugely wins: A's change is 63.117212459956036

A much lower ELO, B barely wins: A's change is 12.0
A much lower ELO, A barely wins: A's change is 36.0
A much lower ELO, B hugely wins: A's change is -36.0
A much lower ELO, A hugely 

In [5]:
prev_elo_time_table = pd.concat([players_table.player, pd.DataFrame(prev_elo_time)], axis=1).T
prev_elo_time_table.columns = players_table.player
prev_elo_time_table = prev_elo_time_table.iloc[2:,:]
prev_elo_time_table = prev_elo_time_table.map(lambda x: x if x > 1 else 0.0)

prev_other_elo_time_table = pd.concat([players_table.player, pd.DataFrame(prev_other_elo_time)], axis=1).T
prev_other_elo_time_table.columns = players_table.player
prev_other_elo_time_table = prev_other_elo_time_table.iloc[2:,:]
prev_other_elo_time_table = prev_other_elo_time_table.map(lambda x: x if x > 1 else 0.0)

elo_time_table = pd.concat([players_table.player, pd.DataFrame(elo_time)], axis=1).T
elo_time_table.columns = players_table.player
elo_time_table = elo_time_table.iloc[2:,:]
elo_time_table.replace(0.0, np.nan, inplace=True)

d_elo_time_table = pd.concat([players_table.player, pd.DataFrame(d_elo_time)], axis=1).T
d_elo_time_table.columns = players_table.player
d_elo_time_table = d_elo_time_table.iloc[2:,:]

ec_time_table = pd.concat([players_table.player, pd.DataFrame(ec_time)], axis=1).T
ec_time_table.columns = players_table.player
ec_time_table = ec_time_table.iloc[2:,:]

caa_time_table = pd.concat([players_table.player, pd.DataFrame(np.round(caa_time, decimals=5))], axis=1).T
caa_time_table.columns = players_table.player
caa_time_table = caa_time_table.iloc[2:,:]

wp_time_table = pd.concat([players_table.player, pd.DataFrame(np.round(wp_time, decimals=5))], axis=1).T
wp_time_table.columns = players_table.player
wp_time_table = wp_time_table.iloc[2:,:]

# adding current elo to table and sorting the table!!!
# remember that this is now sorted so all slicing will have to remember that it's sorted
sorted_players_table = players_table.copy()
sorted_players_table['current_elo'] = sorted_players_table['player'].map(elo)
sorted_players_table.sort_values('current_elo', inplace=True, ascending=False)
sorted_players_table.reset_index(drop=True, inplace=True)

with pd.option_context('display.max_rows', 100, 'display.max_columns', None):
    display(prev_elo_time_table)

with pd.option_context('display.max_rows', 100, 'display.max_columns', None):
    display(elo_time_table)

with pd.option_context('display.max_rows', 100, 'display.max_columns', None):
    display(d_elo_time_table)

with pd.option_context('display.max_rows', 100, 'display.max_columns', None):
    display(caa_time_table)

export(elo_time_table, 'elo_time_table', True)

C:\Users\rramirez\AppData\Local\Temp\ipykernel_18292\809325125.py:14: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  elo_time_table.replace(0.0, np.nan, inplace=True)


player,Kristian Banlaoi,Kevin Cooper,Rohan Chowla,Rose Roché,Aaron Carter,Roman Ramirez,Coby Lovelace,Paul Bartenfeld,Jason Jackson,Jack Massingill,Evan Sooklal,Leah Baetcke,Gabe Silverstein,Reagan Fryatt,Carla Betancourt,Kevin Lee,Cason Duszak,Will Simpson,Ann Hall,Helen Dunn,Noah Dale,Yvonne Nguyen,Anna Brown,Brian Tafazoli,Sam Tellis,Nathan Snow,Piper Parker,Matthew Rusten,Cassie Deering,Yafu LastName,Alex LastName,Luci Nguyen,Eric Edwards,Kim LastName,Julie Jackson,Carolyn LastName,Matthew Werfel,Autumn Hong,Milen Satish,Luke Woods,Matthew Kish,Connor Rogers,Rasmi Tangirala,Alexis Alms,Charles Myersberg,Bryce Taylor,Ronan McKinney,Ryan Healy,Graham Longhofer,Hares Ragub,Nic Faulds,Maria Stalcup,Benjamin Merkel,Sakeena Saidou,Harper Willingham,Z Hayes,Victoria LastName
1,0.0,0.0,0.0,1200.0,1200.0,0.0,0.0,0.0,0.0,0.0,1200.0,0.0,0.0,0.0,0.0,0.0,0.000000,0.0,0.0,0.0,0.0,0.0,0.0,1200.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.000000,0.0,0.000000,0.00000,0.0,0.0,0.0,0.000000,0.000000,0.000000,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.000000,0.0
2,0.0,0.0,0.0,0.0,0.0,0.0,1200.0,0.0,1200.0,1200.0,0.0,0.0,0.0,0.0,0.0,0.0,0.000000,1200.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.000000,0.0,0.000000,0.00000,0.0,0.0,0.0,0.000000,0.000000,0.000000,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.000000,0.0
3,0.0,1200.0,0.0,0.0,0.0,1200.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.000000,0.0,1200.0,0.0,0.0,0.0,1200.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.000000,0.0,0.000000,0.00000,0.0,0.0,0.0,0.000000,0.000000,0.000000,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.000000,0.0
4,0.0,0.0,1200.0,0.0,0.0,0.0,0.0,1200.0,0.0,0.0,0.0,1200.0,0.0,0.0,0.0,1200.0,0.000000,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.000000,0.0,0.000000,0.00000,0.0,0.0,0.0,0.000000,0.000000,0.000000,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.000000,0.0
5,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1164.0,1164.0,0.0,0.0,0.0,0.0,0.0,0.000000,1164.0,0.0,0.0,0.0,0.0,0.0,1164.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.000000,0.0,0.000000,0.00000,0.0,0.0,0.0,0.000000,0.000000,0.000000,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.000000,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
182,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1186.476939,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.000000,0.0,0.000000,0.00000,0.0,0.0,0.0,1192.800993,1201.460545,0.000000,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1214.961128,0.0
183,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.000000,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1327.713715,0.0,1253.150813,1239.48367,0.0,0.0,0.0,0.000000,0.000000,1305.294944,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.000000,0.0
184,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1235.859191,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.000000,0.0,1246.644192,1232.97705,0.0,0.0,0.0,1242.183244,0.000000,0.000000,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.000000,0.0
185,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1247.284890,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1334.220335,0.0,0.000000,0.00000,0.0,0.0,0.0,1253.608944,0.000000,1311.801564,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.000000,0.0


player,Kristian Banlaoi,Kevin Cooper,Rohan Chowla,Rose Roché,Aaron Carter,Roman Ramirez,Coby Lovelace,Paul Bartenfeld,Jason Jackson,Jack Massingill,Evan Sooklal,Leah Baetcke,Gabe Silverstein,Reagan Fryatt,Carla Betancourt,Kevin Lee,Cason Duszak,Will Simpson,Ann Hall,Helen Dunn,Noah Dale,Yvonne Nguyen,Anna Brown,Brian Tafazoli,Sam Tellis,Nathan Snow,Piper Parker,Matthew Rusten,Cassie Deering,Yafu LastName,Alex LastName,Luci Nguyen,Eric Edwards,Kim LastName,Julie Jackson,Carolyn LastName,Matthew Werfel,Autumn Hong,Milen Satish,Luke Woods,Matthew Kish,Connor Rogers,Rasmi Tangirala,Alexis Alms,Charles Myersberg,Bryce Taylor,Ronan McKinney,Ryan Healy,Graham Longhofer,Hares Ragub,Nic Faulds,Maria Stalcup,Benjamin Merkel,Sakeena Saidou,Harper Willingham,Z Hayes,Victoria LastName
1,NaN,NaN,NaN,1236.0,1236.0,NaN,NaN,NaN,NaN,NaN,1164.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1164.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,NaN,NaN,NaN,NaN,NaN,NaN,1236.0,NaN,1236.0,1164.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1164.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,NaN,1236.0,NaN,NaN,NaN,1236.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1164.0,NaN,NaN,NaN,1164.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,NaN,NaN,1272.0,NaN,NaN,NaN,NaN,1128.0,NaN,NaN,NaN,1128.0,NaN,NaN,NaN,1272.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
5,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1188.0,1140.0,NaN,NaN,NaN,NaN,NaN,NaN,1188.0,NaN,NaN,NaN,NaN,NaN,1140.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
182,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1235.859191,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1248.507298,1152.078293,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1179.079459,NaN
183,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1356.639106,NaN,1246.644192,1219.309907,NaN,NaN,NaN,NaN,NaN,1311.801564,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
184,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1247.284890,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1235.218493,1207.884207,NaN,NaN,NaN,1259.932998,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
185,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1288.305529,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1315.618467,NaN,NaN,NaN,NaN,NaN,NaN,1300.953637,NaN,1270.780925,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


player,Kristian Banlaoi,Kevin Cooper,Rohan Chowla,Rose Roché,Aaron Carter,Roman Ramirez,Coby Lovelace,Paul Bartenfeld,Jason Jackson,Jack Massingill,Evan Sooklal,Leah Baetcke,Gabe Silverstein,Reagan Fryatt,Carla Betancourt,Kevin Lee,Cason Duszak,Will Simpson,Ann Hall,Helen Dunn,Noah Dale,Yvonne Nguyen,Anna Brown,Brian Tafazoli,Sam Tellis,Nathan Snow,Piper Parker,Matthew Rusten,Cassie Deering,Yafu LastName,Alex LastName,Luci Nguyen,Eric Edwards,Kim LastName,Julie Jackson,Carolyn LastName,Matthew Werfel,Autumn Hong,Milen Satish,Luke Woods,Matthew Kish,Connor Rogers,Rasmi Tangirala,Alexis Alms,Charles Myersberg,Bryce Taylor,Ronan McKinney,Ryan Healy,Graham Longhofer,Hares Ragub,Nic Faulds,Maria Stalcup,Benjamin Merkel,Sakeena Saidou,Harper Willingham,Z Hayes,Victoria LastName
1,0.0,0.0,0.0,36.0,36.0,0.0,0.0,0.0,0.0,0.0,-36.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,-36.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2,1333604319588983000677835224031043897530372730...,0.0,0.0,0.0,0.0,0.0,36.0,0.0,36.0,-36.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,-36.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
3,0.0,36.0,0.0,0.0,0.0,36.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,-36.0,0.0,0.0,0.0,-36.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
4,0.0,0.0,72.0,0.0,0.0,0.0,0.0,-72.0,0.0,0.0,0.0,-72.0,0.0,0.0,0.0,72.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
5,-3384607070172718636013587823478436408327112211...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,24.0,-24.0,0.0,0.0,0.0,0.0,0.0,0.0,24.0,0.0,0.0,0.0,0.0,0.0,-24.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
182,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,49.382251,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,49.382251,-49.382251,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,-49.382251,0.0
183,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,6.50662,0.0,-6.50662,-6.50662,0.0,0.0,0.0,0.0,0.0,6.50662,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
184,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,11.4257,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,-11.4257,-11.4257,0.0,0.0,0.0,11.4257,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
185,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,41.020639,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,-41.020639,0.0,0.0,0.0,0.0,0.0,0.0,41.020639,0.0,-41.020639,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


player,Kristian Banlaoi,Kevin Cooper,Rohan Chowla,Rose Roché,Aaron Carter,Roman Ramirez,Coby Lovelace,Paul Bartenfeld,Jason Jackson,Jack Massingill,Evan Sooklal,Leah Baetcke,Gabe Silverstein,Reagan Fryatt,Carla Betancourt,Kevin Lee,Cason Duszak,Will Simpson,Ann Hall,Helen Dunn,Noah Dale,Yvonne Nguyen,Anna Brown,Brian Tafazoli,Sam Tellis,Nathan Snow,Piper Parker,Matthew Rusten,Cassie Deering,Yafu LastName,Alex LastName,Luci Nguyen,Eric Edwards,Kim LastName,Julie Jackson,Carolyn LastName,Matthew Werfel,Autumn Hong,Milen Satish,Luke Woods,Matthew Kish,Connor Rogers,Rasmi Tangirala,Alexis Alms,Charles Myersberg,Bryce Taylor,Ronan McKinney,Ryan Healy,Graham Longhofer,Hares Ragub,Nic Faulds,Maria Stalcup,Benjamin Merkel,Sakeena Saidou,Harper Willingham,Z Hayes,Victoria LastName
1,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
3,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
4,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
5,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,-0.18709,-0.18709,0.0,0.0,0.0,0.0,0.0,0.0,-0.18709,0.0,0.0,0.0,0.0,0.0,-0.18709,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
182,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,-0.03742,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,-0.03742,0.07777,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.07777,0.0
183,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.66124,0.0,0.20518,0.20518,0.0,0.0,0.0,0.0,0.0,0.66124,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
184,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.2192,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.17139,0.17139,0.0,0.0,0.0,0.2192,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
185,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.27849,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.69463,0.0,0.0,0.0,0.0,0.0,0.0,0.27849,0.0,0.69463,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


In [6]:
import plotly.graph_objects as go
import plotly.colors as pc

average_elo = 1200
cup_diffs = [-2, -1, -0.5, -0.25, 0, 0.25, 0.5, 1, 2]
expected_elos = [average_elo - expected_score_inverse(x) for x in cup_diffs]

fig = go.Figure()

# CAA is cups against average (like WAR)
# how many cups will you make against the average player?

hrect_colors = pc.sequential.Turbo
hrect_annots = ['-1.00 CAA','-0.50 CAA','-0.25 CAA','0.00 CAA','+0.25 CAA','+0.50 CAA', '+1.00 CAA', '+2.00 CAA']
for i in range(len(expected_elos) - 1):
    fig.add_hrect(
        annotation_position="top left",
        y0=expected_elos[i],
        y1=expected_elos[i+1],
        fillcolor=hrect_colors[i % len(hrect_colors)],
        opacity=0,
        line_width=0,
        annotation=dict(
            text=f'<b>{hrect_annots[i]}</b>',
            font=dict(
                size=7
            )
        )
    )

ranking = 1
trace_colors = pc.qualitative.Light24
for (i, (player,current_rating)) in enumerate(sorted(elo.items(), key=lambda x:x[1], reverse=True)):
    fig.add_trace(go.Scatter(
        x=elo_time_table.index,
        y=elo_time_table[player],
        name=f'#{ranking} ({current_rating:.0f}) {player}',
        mode='lines+markers',
        connectgaps=True,
        text=[f"<br><b>Before-Game ELO: </b>{x[2]:.0f}<br><b>Expected Cups Against Opponent: </b>{x[0]:+.2f}<br><b>Expected Cups Against Average: </b>{x[1]:+.2f}<br><br><b>Change in ELO:</b> {x[3]:+.0f}<br>" for x in zip(
                ec_time_table[player],
                caa_time_table[player],
                prev_elo_time_table[player],
                d_elo_time_table[player]
            )
        ],
        # text=[f"<b>Change in ELO:</b> {c:.0f}<br>" for c in d_elo_time_table[player]],
        line=dict(
            shape = 'hv',
            color=trace_colors[i % len(trace_colors)]
        )
        # maybe add a per-match section of expected score diff
    ))
    ranking += 1

for (i, season) in enumerate(seasons):
    fig.add_vrect(
        annotation_text=season_labels[i],
        annotation_position="top left",
        x0=matches_table['event'][matches_table['event'] == season].index[0] + 0.5,
        x1=matches_table['event'][matches_table['event'] == season].index[-1] + 1.5,
        fillcolor=vrect_colors[i % len(vrect_colors)],
        opacity=0.1,
        line_width=0,
    )

dropdown_labels = [
    "All",
    "2019 Season",
    "2020 Season",
    "2021 Season",
    "2022 Season",
    "2023 Season",
    "Snares",
    "Tenors",
    "Basses",
    "Cymbals",
    "Pit/Cymbals",
    "Non-Marchers",
    "2022 Grads",
    "2023 Grads",
    "2024 Grads",
    "2025 Grads",
    "2026 Grads",
    "2027 Grads",
    "2021 Doubles",
    "2022 Singles",
    "2022 Doubles",
    "2023 Singles",
    "2023 Doubles",
    "2024 Singles",
    "2024 Doubles" #,
    # "2023 Beach Week"
]
dropdown_categories = [
    [True for _ in sorted_players_table['player']],
    [((type(x) != float)&(x != 'non-marcher')) for x in sorted_players_table['2019']],
    [((type(x) != float)&(x != 'non-marcher')) for x in sorted_players_table['2020']],
    [((type(x) != float)&(x != 'non-marcher')) for x in sorted_players_table['2021']],
    [((type(x) != float)&(x != 'non-marcher')) for x in sorted_players_table['2022']],
    [((type(x) != float)&(x != 'non-marcher')) for x in sorted_players_table['2023']],
    [x > 0 for x in sorted_players_table[['2019', '2020', '2021', '2022', '2023']].apply(lambda x: x == 'snare').sum(axis=1)],
    [x > 0 for x in sorted_players_table[['2019', '2020', '2021', '2022', '2023']].apply(lambda x: x == 'tenors').sum(axis=1)],
    [x > 0 for x in sorted_players_table[['2019', '2020', '2021', '2022', '2023']].apply(lambda x: x == 'bass').sum(axis=1)],
    [x > 0 for x in sorted_players_table[['2019', '2020', '2021', '2022', '2023']].apply(lambda x: x == 'cymbals').sum(axis=1)],
    [x > 0 for x in sorted_players_table[['2019', '2020', '2021', '2022', '2023']].apply(lambda x: x == 'pit-cymbals').sum(axis=1)],
    [x > 0 for x in sorted_players_table[['2019', '2020', '2021', '2022', '2023']].apply(lambda x: x == 'non-marcher').sum(axis=1)],
    [x == '2022' for x in sorted_players_table['grad_year']],
    [x == '2023' for x in sorted_players_table['grad_year']],
    [x == '2024' for x in sorted_players_table['grad_year']],
    [x == '2025' for x in sorted_players_table['grad_year']],
    [x == '2026' for x in sorted_players_table['grad_year']],
    [x == '2027' for x in sorted_players_table['grad_year']],
    [not np.isnan(x) for x in sorted_players_table['seed_2021_2s']],
    [not np.isnan(x) for x in sorted_players_table['seed_2022_1s']],
    [not np.isnan(x) for x in sorted_players_table['seed_2022_2s']],
    [not np.isnan(x) for x in sorted_players_table['seed_2023_1s']],
    [not np.isnan(x) for x in sorted_players_table['seed_2023_2s']],
    [not np.isnan(x) for x in sorted_players_table['seed_2024_1s']],
    [not np.isnan(x) for x in sorted_players_table['seed_2024_2s']]
    # [x == True for x in sorted_players_table['beach_week_2023']]
]

def active_ranking(cat):
    i = 0
    t = 0
    retList = list()
    for c in cat:
        retList.append(f"#{i+1} ({sorted_players_table['current_elo'][t]:.0f}) {sorted_players_table['player'][t]}")
        if c is True:
            i += 1
        t += 1
    return retList 

dropdown_dicts = [
    dict(
    label=label,
    method='restyle',
    args=[{"visible": category, "name": active_ranking(category)}]
    ) for (label, category) in zip(dropdown_labels, dropdown_categories)
]


fig.update_layout(
    title=f'<b>The UVA Drumline Pong ELO Rating System by Roman Ramirez</b><br><i>Updated: {str(datetime.datetime.now().strftime("%A, %b %d, %Y %H:%M:%S"))}',#</i> *Note that non-tournament games were recorded only after 2023 Singles.',
    xaxis_title='<b>Game Number</b>',
    yaxis_title='<b>ELO Rating</b>',

    updatemenus=[
        dict(
            active=0,
            buttons=dropdown_dicts
            ,
        )       
    ]
)

team1_list = list()
team2_list = list()
for (i, row) in matches_table.iterrows():
    if type(row.t1p2) is not str:
        team1_list.append(row.t1p1)
    else:
        team1_list.append(row.t1p1 + " and " + row.t1p2)

    if type(row.t2p2) is not str:
        team2_list.append(row.t2p1)
    else:
        team2_list.append(row.t2p1 + " and " + row.t2p2)

customdata = np.stack((team1_list, team2_list, matches_table['score1'], matches_table['score2'], matches_table['date'], matches_table['match_number']), axis=-1)
hovertemplate = (
    '<i>%{customdata[4]|%A, %B %d, %Y}, Game %{customdata[5]} (%{x})</i><br>' +
    '<b>%{fullData.name}</b><br><br>' + 
    '<b>%{customdata[0]} vs. %{customdata[1]}</b><br>' +
    '<b>Final Score:</b> %{customdata[2]}-%{customdata[3]}<br>' + 
    '%{text}' + 
    '<b>After-Game ELO:</b> %{y:,.0f}<br>' +
    '<extra></extra>'
)

fig.update_traces(
    customdata=customdata,
    hovertemplate=hovertemplate,
    opacity=0.8,
    legendgrouptitle_text='<b>#<i>Rank</i> (<i>Current ELO</i>) <i>Player</i></b>'
)

for (cd, eo) in zip(cup_diffs, expected_elos):
    fig.add_shape(
        type='line',
        x0=elo_time_table.index.min() - 5,
        y0=eo,
        x1=elo_time_table.index.max() + 5,
        y1=eo,
        line=dict(
            color='black',
            dash='dot'
        ),
        opacity=0.10,
    )
    # fig.add_trace(
    #     go.Scatter(
    #         x=[elo_time_table.index.min() - 5],
    #         y=[eo],
    #         text=f"{cd} cups",
    #         mode="text",
    #         textfont=dict(
    #             size=7,
    #         ),
    #         showlegend=False
    #     )
    # )

fig.show()
fig.write_html("index.html")

In [7]:
season_avg_elo_dict = dict()
season_med_elo_dict = dict()
season_strength_dict = dict()
for (i, season) in enumerate(seasons):
    season_label = season_labels[i].replace("<br>", " ")
    x1 = matches_table['event'][matches_table['event'] == season].index[0] + 0.5
    x2 = matches_table['event'][matches_table['event'] == season].index[-1] + 1.5
    
    elo_time_table_per_season = elo_time_table.loc[(elo_time_table.index > x1) & (elo_time_table.index < x2),:]
    elo_time_table_per_season = elo_time_table_per_season.loc[:,elo_time_table_per_season.notna().any()]
    # axis = 0 because we're finding average over all players, not over all game numbers
    y_avg = elo_time_table_per_season.ffill().mean(axis=0).mean()
    y_med = elo_time_table_per_season.ffill().median(axis=0).median()

    season_avg_elo_dict.update({season_label: y_avg})
    season_med_elo_dict.update({season_label: y_med})

    season_player_table = sorted_players_table.loc[[x in elo_time_table_per_season.columns for x in sorted_players_table['player']],:]
    season_strength_dict.update({season_label: season_player_table['current_elo'].mean()})

season_elo_fig = go.Figure()

season_elo_fig.add_trace(go.Scatter(
    x=list(season_avg_elo_dict.keys()),
    y=list(season_avg_elo_dict.values()),
    connectgaps=True,
    name="Average"
))
season_elo_fig.add_trace(go.Scatter(
    x=list(season_med_elo_dict.keys()),
    y=list(season_med_elo_dict.values()),
    connectgaps=True,
    name="Median"
))
season_elo_fig.add_trace(go.Scatter(
    x=list(season_strength_dict.keys()),
    y=list(season_strength_dict.values()),
    connectgaps=True,
    name="Power"
))
season_elo_fig.update_layout(
    xaxis_title="Season",
    yaxis_title="ELO",
    title="Statistics of Active Players Per Season",
    showlegend=True,
)

season_elo_fig.show()

### Power ###
# This retrospective measure represents the average strength of the league at the time
# Measure by finding all the players who participated in that tournament, and taking their respective ELO when they retired, and finding the average of that

### Average ###
# This measure represents the average current ELO of the tournament
# Measure by finding all the players who participated in that tournament, finding each player's average active ELO in the tournament, and then finding the average of that 
# Measure over time shows "power creep"

### CONCLUSIONS ###
# If Power > Average, then the tournament was more difficult to play than the ELO is portraying
# If Power = Average, then the tournament was as difficult to play as the ELO is portraying 
# If Power < Average, then the tournament was easier to play as the ELO is portraying 
# If Average > 1200, then that tournament has more highly-rated players competing

In [8]:
set_players = set(players_table['player'])

# player stats

player_stats_table = pd.DataFrame(np.zeros([len(set_players), 0]))
player_stats_table.index = set_players
player_stats_table.sort_index(axis=0, key=(lambda x: [elo[y] for y in x.values]), ascending=False, inplace=True)

# ELO

player_stats_table['Rank'] = 0
player_stats_table['cELO'] = ([elo[x] for x in player_stats_table.index])
player_stats_table['Rank'] = player_stats_table['cELO'].rank(ascending=False)
player_stats_table['cELO'] = player_stats_table['cELO']
player_stats_table['maxELO'] = elo_time_table.replace(to_replace=np.nan, value=-np.inf).max(axis=0)
player_stats_table['minELO'] = elo_time_table.replace(to_replace=np.nan, value=np.inf).min(axis=0)

date2matches_len = dict.fromkeys(set(matches_table['date']),0)
date2matches_first = dict.fromkeys(set(matches_table['date']),0)
for (i, row) in matches_table.iterrows():
    if date2matches_len[row['date']] == 0:
        date2matches_first[row['date']] = i
    date2matches_len[row['date']] += 1

# CHAMPIONSHIPS AND RUNNER UPS, need to add doubles functionality

# RECORDS
singles = ['t1p1','t2p1']
doubles = ['t1p2','t2p2']
t1s = ['t1p1','t1p2']
t2s = ['t2p1','t2p2']

player_stats_table.loc[:,'1st'] = 0
player_stats_table.loc[:,'2nd'] = 0

for date in set(matches_table['date']):
    row = matches_table.iloc[date2matches_first[date] + date2matches_len[date] - 1, :]
    if row['score1'] > row['score2']:
        for t in t1s:
            player_stats_table.loc[player_stats_table.index == row[t],'1st'] += 1
        for t in t2s:
            player_stats_table.loc[player_stats_table.index == row[t],'2nd'] += 1
    elif row['score2'] > row['score1']:
        for t in t2s:
            player_stats_table.loc[player_stats_table.index == row[t],'1st'] += 1
        for t in t1s:
            player_stats_table.loc[player_stats_table.index == row[t],'2nd'] += 1
        

# initilization
player_stats_table.loc[:,  'RW'] = 0
player_stats_table.loc[:,  'RL'] = 0
player_stats_table.loc[:, 'OTW'] = 0
player_stats_table.loc[:, 'OTL'] = 0
player_stats_table.loc[:,  'TP'] = 0
player_stats_table.loc[:,  'GP'] = 0
player_stats_table.loc[:,   'W'] = 0
player_stats_table.loc[:,   'L'] = 0
player_stats_table.loc[:, 'PTS'] = 0
player_stats_table.loc[:,  'CF'] = 0
player_stats_table.loc[:,  'CA'] = 0
player_stats_table.loc[:, '+/-'] = 0
player_stats_table.loc[:,  'CFg'] = 0
player_stats_table.loc[:,  'CAg'] = 0
player_stats_table.loc[:, '+/-g'] = 0
player_stats_table.loc[:,  'CFt'] = 0
player_stats_table.loc[:,  'CAt'] = 0
player_stats_table.loc[:, '+/-t'] = 0
player_stats_table['tCAA'] = caa_time_table.sum(axis=0).apply(float)
player_stats_table.loc[:,'aCAA'] = 0

tourneys_per_player = dict()
for p in player_stats_table.index:
    tourneys_per_player.update({p: set()})

for (i, row) in matches_table.iterrows():

    Ws = ['W']
    Ls = ['L']

    # calcs
    is_t1_win = row['score1'] > row['score2']
    is_ot = max(row['score1'], row['score2']) > 6
    Ws = Ws + ['OTW'] if is_ot else Ws + ['RW']
    Ls = Ls + ['OTL'] if is_ot else Ls + ['RL']

    # GP
    for p in (singles + doubles):
        player_stats_table.loc[player_stats_table.index == row[p],'GP'] += 1

    # TP, tournaments played
    for p in (singles + doubles):
        if not row[p] != row[p]:
            placeholder_set = tourneys_per_player[row[p]]
            placeholder_set.add(row['event'])
            tourneys_per_player.update({row[p]: placeholder_set})

    # W and L
    for t in t1s:
        for W in Ws:
            player_stats_table.loc[player_stats_table.index == row[t],W] += is_t1_win
        for L in Ls:
            player_stats_table.loc[player_stats_table.index == row[t],L] += not is_t1_win
        player_stats_table.loc[player_stats_table.index == row[t],'CF'] += row['score1']
        player_stats_table.loc[player_stats_table.index == row[t],'CA'] += row['score2']
    for t in t2s:
        for W in Ws:
            player_stats_table.loc[player_stats_table.index == row[t],W] += not is_t1_win
        for L in Ls:
            player_stats_table.loc[player_stats_table.index == row[t],L] += is_t1_win
        player_stats_table.loc[player_stats_table.index == row[t],'CA'] += row['score1']
        player_stats_table.loc[player_stats_table.index == row[t],'CF'] += row['score2']

player_stats_table['PTS'] = (player_stats_table['W'] * 2) + player_stats_table['OTL']
player_stats_table['+/-'] = (player_stats_table['CF'] - player_stats_table['CA'])
player_stats_table['aCAA'] = (player_stats_table['tCAA'].apply(float) / player_stats_table['GP'])
player_stats_table['TP'] = {k: len(v) for (k, v) in tourneys_per_player.items()}.values()
player_stats_table['PCT'] = (player_stats_table['W'] / player_stats_table['GP'])
    
player_stats_table['CFg'] = (player_stats_table['CF'] / player_stats_table['GP'])
player_stats_table['CAg'] = (player_stats_table['CA'] / player_stats_table['GP'])
player_stats_table['CFt'] = (player_stats_table['CF'] / player_stats_table['TP'])
player_stats_table['CAt'] = (player_stats_table['CA'] / player_stats_table['TP'])

player_stats_table['+/-g'] = (player_stats_table['CFg'] - player_stats_table['CAg'])
player_stats_table['+/-t'] = (player_stats_table['CFt'] - player_stats_table['CAt'])

def highlight_max(s):
    if (s.dtype == np.int64) or (s.dtype == np.float64): 
        is_max = s == s.max() 
    else: 
        is_max = [False for _ in range(s.shape[0])] 
    return ['font-weight:bold' if cell else '' for cell in is_max] 

def highlight_min(s):
    if (s.dtype == np.int64) or (s.dtype == np.float64): 
        is_min = s == s.min() 
    else: 
        is_min = [False for _ in range(s.shape[0])] 
    return ['font-style:italic' if cell else '' for cell in is_min] 

from pandas.io.formats.style import Styler

def init_styler(df):
    sdf = Styler(df, cell_ids=False)

    styles = [
        {
            'selector': '',
            'props': [
                ('text-align', 'center'),
                ('font-size', '11pt'),
                ('border-collapse', 'collapse'),#'separated'),
                ('border', '1px solid gray')
            ]
        },
        {
            'selector': 'caption',
            'props': [('font', 'bold 20px Verdana')]
        },
        {
            'selector': 'tr:nth-child(odd)',
            'props': 'background-color: #D3D3D3'
        },
        {
            'selector': 'tr:hover',
            'props': [('background-color','#add8e6')]
        },
        {
            'selector': 'td:hover',
            'props': [('background-color','#ffffb3')]
        },
        {
            'selector': 'th',
            'props': [
                ('font-family', 'Verdana'),
                ('font-weight', 'normal'),
                ('padding-left', '5px'),
                ('padding-right', '5px'),
            ]
        },
        {
            'selector': 'td',
            'props': [
                ('font-family', 'Courier New'),
                ('padding-left', '5px'),
                ('padding-right', '5px'),
                ('text-align','right')
            ]
        },
    ] + [
        {'selector': f'th.col{x}', 'props': 'border-bottom: 1px solid gray'} for x in range(player_stats_table.shape[1])
    ] + [
        {'selector': f'th.row{x}', 'props': [('text-align', 'right'), ('border-right', '1px solid gray')]} for x in range(player_stats_table.shape[0])
    ] + [
        {'selector': f'th.col{x}', 'props': [('border-left', '1px solid gray')]} for x in range(1)
    ] + [
        {'selector': f'th.row{x}', 'props': [('border-top', '1px solid gray')]} for x in range(1)
    ]
    sdf.set_table_styles(styles, overwrite=True)
    sdf = sdf.apply(highlight_max, axis=0)
    sdf = sdf.apply(highlight_min, axis=0)  

    return sdf

player_stats_table_styled = init_styler(player_stats_table)
player_stats_table_styled.set_caption("Player Table Statistics")
player_stats_table_styled = player_stats_table_styled.format(
    {x: '{:.0f}' for x in ['Rank','cELO','maxELO','minELO']} |
    {x: '{:+.0f}' for x in ['+/-']} |
    {x: '{:+.1f}' for x in ['+/-g','+/-t']} |
    {x: '{:.2f}' for x in ['aCAA']} |
    {x: '{:.1f}' for x in ['tCAA','CFg','CAg','CFt','CAt']} | 
    {x: '{:.3f}' for x in ['PCT']}
)
# player_stats_table_styled.set_table_styles({
#     "Rank":  {"selector": ''}
    
# }, overwrite=False, axis=0)
# player_stats_table_styled.apply(lambda x: ['border-right: 1px solid black' if x.name == 'Rank' else '' for v in x])
player_stats_table_styled.set_table_styles({
        x: [{
            'selector': 'th,td',
            'props': 'border-right: 1px solid gray'
        }] for x in ['Rank', 'minELO', '2nd', 'RL', 'OTL', 'GP', 'PTS', '+/-', '+/-g', '+/-t', 'aCAA']
    }, overwrite=False, axis=0)
display(player_stats_table_styled)
player_stats_table_styled.to_html('exports/player_stats_table.html')


,Rank,cELO,maxELO,minELO,1st,2nd,RW,RL,OTW,OTL,TP,GP,W,L,PTS,CF,CA,+/-,CFg,CAg,+/-g,CFt,CAt,+/-t,tCAA,aCAA,PCT
Kevin Lee,1,1571,1571,1268,5,2,25,5,2,0,7,32,27,5,54,192,138,+54,6.0,4.3,+1.7,27.4,19.7,+7.7,32.0,1.00,0.844
Aaron Carter,2,1385,1385,1155,1,0,8,7,3,0,4,18,11,7,22,103,90,+13,5.7,5.0,+0.7,25.8,22.5,+3.2,1.1,0.06,0.611
Roman Ramirez,3,1376,1462,1211,2,0,14,5,1,1,4,21,15,6,31,124,103,+21,5.9,4.9,+1.0,31.0,25.8,+5.2,11.0,0.52,0.714
Milen Satish,4,1352,1357,1188,1,1,8,4,0,1,3,13,8,5,17,69,56,+13,5.3,4.3,+1.0,23.0,18.7,+4.3,4.0,0.31,0.615
Rohan Chowla,5,1339,1389,1268,3,2,21,9,5,2,8,37,26,11,54,224,195,+29,6.1,5.3,+0.8,28.0,24.4,+3.6,26.6,0.72,0.703
Ryan Healy,6,1307,1312,1247,1,0,4,1,0,0,1,5,4,1,8,27,17,+10,5.4,3.4,+2.0,27.0,17.0,+10.0,2.3,0.46,0.800
Kevin Cooper,7,1272,1295,1198,1,0,10,5,0,0,3,15,10,5,20,79,71,+8,5.3,4.7,+0.5,26.3,23.7,+2.7,5.5,0.37,0.667
Autumn Hong,8,1272,1278,1205,0,1,6,4,1,0,2,11,7,4,14,61,61,+0,5.5,5.5,+0.0,30.5,30.5,+0.0,1.8,0.17,0.636
Nathan Snow,9,1270,1282,1234,0,0,5,3,1,1,2,10,6,4,13,60,55,+5,6.0,5.5,+0.5,30.0,27.5,+2.5,1.5,0.15,0.600
Bryce Taylor,10,1265,1301,1184,0,1,5,4,2,0,2,11,7,4,14,61,56,+5,5.5,5.1,+0.5,30.5,28.0,+2.5,1.3,0.12,0.636


In [9]:
def full2shortname(name, no_last="LastName"):
    split = name.split(" ")
    if len(split) != 2:
        return None
    (first, last) = split
    if last == no_last:
        return first
    return first+" "+last[0]



def create_player_table(name):
    matches_per_player_table = matches_table.loc[[name in list(x) for (i, x) in matches_table[['t1p1','t1p2','t2p1','t2p2']].iterrows()]]

    # swapping so that yourself is always team 1
    
    for (i, row) in matches_per_player_table.iterrows():
        if (name == row['t2p1']) or (name == row['t2p2']):
            matches_per_player_table.loc[i,'t1p1'], matches_per_player_table.loc[i,'t2p1'] = matches_per_player_table.loc[i,'t2p1'], matches_per_player_table.loc[i,'t1p1']
            matches_per_player_table.loc[i,'t1p2'], matches_per_player_table.loc[i,'t2p2'] = matches_per_player_table.loc[i,'t2p2'], matches_per_player_table.loc[i,'t1p2']
            matches_per_player_table.loc[i,'score1'], matches_per_player_table.loc[i,'score2'] = matches_per_player_table.loc[i,'score2'], matches_per_player_table.loc[i,'score1']
    for (i, row) in matches_per_player_table.iterrows():
        if (name == row['t1p2']):
            matches_per_player_table.loc[i,'t1p1'], matches_per_player_table.loc[i,'t1p2'] = matches_per_player_table.loc[i,'t1p2'], matches_per_player_table.loc[i,'t1p1']

    player_table = pd.DataFrame()

    matches_per_player_table.index += 1
    player_table.index += 1

    def swap(x):
        (a, b) = x.split("_")
        return b+"_"+a

    player_table['Date'] = matches_per_player_table['date'].apply(lambda x: pd.to_datetime(x).strftime('%m/%d/%y'))
    player_table['Event'] = matches_per_player_table['event']
    player_table['MN'] = matches_per_player_table['match_number']
    player_table['Teammate'] = [full2shortname(y) if y == y else "" for (x, y) in zip(matches_per_player_table['t1p1'], matches_per_player_table['t1p2'])]
    player_table['Seed'] = ["("+str(int(y))+")" for y in [players_table.loc[players_table['player'] == name, f'seed_{x}'].values[0] for x in player_table['Event']]]
    player_table['Opponent(s)'] = [full2shortname(x)+", "+full2shortname(y) if y == y else full2shortname(x) for (x, y) in zip(matches_per_player_table['t2p1'], matches_per_player_table['t2p2'])]
    player_table['oSeed'] = ["("+str(int(y))+")" for y in [players_table.loc[players_table['player'] == y, f'seed_{x}'].values[0] for (x, y) in zip(matches_per_player_table['event'], matches_per_player_table['t2p1'])]]
    
    # player_table['oSeed'] = [int(y) for y in [players_table.loc[players_table['player'] == (z.split(", ")[0]), f'seed_{x}'].values[0] for (x,z) in zip(player_table['Event'],player_table['Opponent(s)'])]]
    player_table['pELO'] = prev_elo_time_table[name].apply(float)
    player_table['poELO'] = prev_other_elo_time_table[name].apply(float)
    player_table['pW'] = wp_time_table[name].apply(float)
    player_table['Result'] = ["W, "+s1+"-"+s2 if s1 > s2 else "L, "+s1+"-"+s2 for (s1, s2) in zip(matches_per_player_table['score1'].apply(str), matches_per_player_table['score2'].apply(str))]
    player_table['dELO'] = d_elo_time_table[name].apply(float)
    player_table['cELO'] = elo_time_table[name].apply(float)
    player_table['EC'] = ec_time_table[name].apply(float)
    player_table['CAA'] = caa_time_table[name].apply(float)
    player_table['STRK'] = None

    # players_table.loc[players_table['player'] == 'Kevin Lee', 'seed_2021_2s']
    
    
    counter = 0
    streak_str = "W0"
    for (i, row) in player_table.iterrows():
        if row['Result'][0] == 'W':
            if streak_str[0] == 'W':
                counter += 1
                streak_str = streak_str[0] + str(counter)
            elif streak_str[0] == 'L':
                streak_str = "W1"
                counter = 1
        if row['Result'][0] == 'L':
            if streak_str[0] == 'L':
                counter += 1
                streak_str = streak_str[0] + str(counter)
            elif streak_str[0] == 'W':
                streak_str = "L1"
                counter = 1
        player_table.loc[i,"STRK"] = streak_str


    player_table_styled = init_styler(player_table)
    player_table_styled.set_caption(f"Per-Game Stats - {name}")
    player_table_styled = player_table_styled.format(
        {x: '{:.0f}' for x in ['cELO', 'dELO','pELO','poELO']} |
        {x: '{:+.0f}' for x in ['dELO']} |
        {x: '{:.3f}' for x in ['EC','CAA']} |
        {x: '{:.0%}' for x in ['pW']}
    )
    player_table_styled.set_table_styles({
        x: [{
            'selector': 'th,td',
            'props': 'border-right: 1px solid gray'
        }] for x in ['MN','Event', 'Seed','oSeed','pW','cELO', 'CAA']
    }, overwrite=False, axis=0)
    
    # player_stats_table_styled.set_table_styles([{
    #     'selector': f'td.row{x}', 'props': [('border-bottom', '1px solid black')]
    # } for x in [4, 10, 16, 20, 24, 27]], overwrite=False, axis=0)

    player_table_styled.apply(lambda x: ['color: red' if 'L' in v else "" for v in x], axis=0, subset=['Result','STRK'])
    player_table_styled.apply(lambda x: ['color: green' if 'W' in v else "" for v in x], axis=0, subset=['Result','STRK'])


    def get_bottom_of_event(df):
        ret_list = list()
        for (a, (i, row)) in enumerate(df.iterrows()):
            if a >= df.shape[0] - 1:
                pass
            elif df.iloc[a,1] != df.iloc[a+1,1]:
                ret_list.append(i)
        # return ['border-bottm: 1px solid gray' if v in ret_list else '' for v in range(ret_list[0])]
        return ret_list
    
    player_table_styled.set_table_styles({
        x: [{
            'selector': 'th,td',
            'props': 'border-bottom: 1px solid gray'
        }] for x in get_bottom_of_event(player_table)
    }, overwrite=False, axis=1)

    player_table_styled.to_html(f'exports/player/{name}_game.html')
    return (player_table_styled, player_table)

for name in player_stats_table.index:
    create_player_table(name)
# display(player_table[['Event','EC','CAA']].groupby(['Event']).sum())

## Simulator
### Given the current state of players' ELOs, simluate a N-player single elimination tournament

In [10]:
# 2: 1, 2
# 3: 1, None, 2, 3
# 4:
# 5:
# 6:
# 7:

In [11]:
# agg_d_elo_table = pd.DataFrame()
# agg_d_elo_table['Avg Change in ELO'] = d_elo_time_table.mean(axis=0)
# agg_d_elo_table['STD Change in ELO'] = d_elo_time_table.std(axis=0) # how streaky is someone, whats the average difference a change will be from the mean change
# agg_d_elo_table.sort_values("STD Change in ELO", ascending=False, inplace=True)

# with pd.option_context('display.max_rows', 100, 'display.max_columns', None):
#     display(pd.DataFrame(agg_d_elo_table))

# import plotly.express as px

# fig_streakiness = px.bar(
#     agg_d_elo_table,
#     y='Avg Change in ELO',
#     error_y='STD Change in ELO',
#     x=agg_d_elo_table.index,
#     title="<b>Streakiness</b>",
#     )

# fig_streakiness.update_layout(
#     title=f'<b>Streakiness by Roman Ramirez</b><br><i>Updated: {str(datetime.datetime.now().strftime("%A, %b %d, %Y %H:%M:%S"))}<i>',
#     xaxis_title='<b>Average Change in ELO</b>',
#     yaxis_title='<b>Player</b>',
# )

# fig_streakiness.show()
# fig_streakiness.write_html("exports/streakiness.html")

In [12]:
# # fig_streak_plot = px.scatter(
# #     agg_d_elo_table,
# #     x='Avg Change in ELO',
# #     y='STD Change in ELO',
# #     text='player'
# # )

# fig_streak_data = go.Scatter(
#     x=agg_d_elo_table['Avg Change in ELO'],
#     y=agg_d_elo_table['STD Change in ELO'],
#     text=agg_d_elo_table.index,
#     mode='markers'
# )

# fig_streak_plot = go.Figure(
#     data=fig_streak_data
# )

# fig_streak_plot.add_shape(
#     type='line',
#     x0=agg_d_elo_table['Avg Change in ELO'].min() * 1.3,
#     y0=agg_d_elo_table['STD Change in ELO'].mean(),
#     x1=agg_d_elo_table['Avg Change in ELO'].max() * 1.3,
#     y1=agg_d_elo_table['STD Change in ELO'].mean(),
#     line=dict(
#         color='black',
#         dash='dot'
#     ),
#     opacity=0.50,
# )

# fig_streak_plot.add_shape(
#     type='line',
#     x0=agg_d_elo_table['Avg Change in ELO'].mean(),
#     y0=agg_d_elo_table['STD Change in ELO'].min() * 1.3,
#     x1=agg_d_elo_table['Avg Change in ELO'].mean(),
#     y1=agg_d_elo_table['STD Change in ELO'].max() * 1.3,
#     line=dict(
#         color='black',
#         dash='dot'
#     ),
#     opacity=0.50,
# )

# fig_streak_plot.update_layout(
#     title=f'<b>Streakiness Plot by Roman Ramirez</b><br><i>Updated: {str(datetime.datetime.now().strftime("%A, %b %d, %Y %H:%M:%S"))}<i>',
#     xaxis_title='<b>Average Change in ELO</b>',
#     yaxis_title='<b>STD Change in ELO</b>',
# )


# fig_streak_plot.show()

In [13]:
# display(expected_score(1200, 1600))

In [14]:
# def expected_score(ratingA, ratingB): # expected difference in cups hit
#     # sigmoid vertical bounds
#     yrange = 12
#     ymin = 6
#     # game variables
#     expected_cups = 2
#     normal_elo_difference = 400

#     # score is the expected difference in cups scored,
#         # if this function spits out -5, then players of those two ratings should end in a difference of 5 cups
#     # this function is a sigmoid function from -6 to 6
#     # this function says an elo difference of "normal_elo_difference" should result in absolute difference in "expected_cups"
#         # this works: an elo difference of 400 should result in a score difference of 2
#     return (yrange / (1 + np.power(expected_cups, (ratingB - ratingA) / normal_elo_difference))) - ymin

# # def expected_cups(ratingA, ratingB):
# #     yrange = 12
# #     ymin = 6
# #     expected_score =